In [ ]:
from classificationhead import CellDINOClassificationHead
import torch

head = CellDINOClassificationHead(input_dim=1024, num_classes=2)

# Example CellDINO global embedding
features = torch.randn(1, 1024)

logits = head(features)

print(logits.shape)

torch.Size([1, 2])


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
from pathlib import Path
import numpy as np

REPO_DIR = Path(r"C:\Users\Wang2_h\dinov2")  # current folder should be the cloned dinov2 repo

WEIGHTS_NAME = "cell_dino_vits8_pretrain_cp-37d20e9c.pth"
WEIGHTS = REPO_DIR / "checkpoints" / WEIGHTS_NAME
MODEL_NAME = "cell_dino_cp_vits8"  # good first choice for HPA-like fluorescence
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class GrayscaleTo5Channel:

    def __call__(self, img):

        """

        Convert PIL image to 5-channel tensor for CellDINO CP model.

        Input:

            PIL image

        Output:

            Tensor [5, H, W], values in [0, 1]

        """

        img = img.convert("L")

        arr = np.asarray(img).astype(np.float32) / 255.0

        arr5 = np.stack([arr, arr, arr, arr, arr], axis=0)

        return torch.from_numpy(arr5)

# ============================================================
# 1. Classification head
# ============================================================

class CellDINOClassificationHead(nn.Module):
    def __init__(self, input_dim=384, num_classes=2, dropout=0.2):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        """
        x: [batch_size, 384]
        return: [batch_size, 2]
        """
        return self.classifier(x)


# ============================================================
# 2. CellDINO classifier wrapper
# ============================================================

class CellDINOClassifier(nn.Module):
    def __init__(
        self,
        celldino_backbone,
        input_dim=384,
        num_classes=2,
        freeze_backbone=True
    ):
        super().__init__()

        self.backbone = celldino_backbone
        self.head = CellDINOClassificationHead(
            input_dim=input_dim,
            num_classes=num_classes
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def extract_features(self, images):
        """
        images: [batch_size, 5, H, W]

        Expected output:
        features: [batch_size, 384]
        """

        # Many DINO-style models have forward_features()
        if hasattr(self.backbone, "forward_features"):
            output = self.backbone.forward_features(images)
        else:
            output = self.backbone(images)

        # Case 1: DINOv2-style dictionary output
        if isinstance(output, dict):
            if "x_norm_clstoken" in output:
                features = output["x_norm_clstoken"]
            elif "cls_token" in output:
                features = output["cls_token"]
            elif "global_embedding" in output:
                features = output["global_embedding"]
            else:
                raise KeyError(
                    f"Cannot find global feature in CellDINO output. "
                    f"Available keys: {output.keys()}"
                )

        # Case 2: direct tensor output
        elif isinstance(output, torch.Tensor):
            features = output

        # Case 3: tuple/list output, assume first element is global feature
        elif isinstance(output, (tuple, list)):
            features = output[0]

        else:
            raise TypeError(f"Unsupported CellDINO output type: {type(output)}")

        # Safety check
        if features.ndim != 2:
            raise ValueError(
                f"Expected features shape [batch_size, feature_dim], "
                f"but got {features.shape}"
            )

        return features

    def forward(self, images):
        """
        images: [batch_size, 5, H, W]
        logits: [batch_size, 2]
        """

        # If backbone is frozen, no need to calculate gradients through CellDINO
        if not any(p.requires_grad for p in self.backbone.parameters()):
            with torch.no_grad():
                features = self.extract_features(images)
        else:
            features = self.extract_features(images)

        logits = self.head(features)
        return logits


# ============================================================
# 3. Data preparation
# ============================================================

image_size = 224
batch_size = 8

train_dir = REPO_DIR / "dataset_singlecell" / "train"
val_dir = REPO_DIR / "dataset_singlecell" / "val"

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=3
    ),

    GrayscaleTo5Channel(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5, 0.5, 0.5],
    )
])
 
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
 
    GrayscaleTo5Channel(),
 
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5, 0.5, 0.5],
    )
])

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=val_dir,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Class mapping:")
print(train_dataset.class_to_idx)


# -----------------------------
# Load Cell-DINO backbone
# -----------------------------

def load_celldino_backbone(repo_dir, model_name, weights_path, device):
    backbone = torch.hub.load(
        str(repo_dir),
        model_name,
        source="local",
        pretrained_path=str(weights_path),
    )

    backbone = backbone.to(device)
    backbone.eval()

    return backbone


celldino_backbone = load_celldino_backbone(
    repo_dir=REPO_DIR,
    model_name=MODEL_NAME,
    weights_path=WEIGHTS,
    device=DEVICE
)



model = CellDINOClassifier(
    celldino_backbone=celldino_backbone,
    input_dim=384,
    num_classes=2,
    freeze_backbone=True
)

model = model.to(DEVICE)


# ============================================================

# 4. Sanity check

# ============================================================

images, labels = next(iter(train_loader))

images = images.to(DEVICE)

labels = labels.to(DEVICE)

model.eval()

with torch.no_grad():

    features = model.extract_features(images)

    logits = model(images)

print("Input batch shape:", images.shape)

print("Feature shape:", features.shape)

print("Logits shape:", logits.shape)

print("Labels shape:", labels.shape)


# ============================================================
# 5. Loss and optimizer
# ============================================================

class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.head.parameters(),   # train only classifier head
    lr=1e-4,
    weight_decay=1e-4
)



# ============================================================
# 6. Training loop
# ============================================================

num_epochs = 50
best_val_acc = -1
SAVE_PATH = REPO_DIR / "best_celldino_symbiont_classifier_cp_vits8.pth"

print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))
print("Train class counts:", np.bincount(train_dataset.targets))
print("Val class counts:", np.bincount(val_dataset.targets))

for epoch in range(num_epochs):
    # -------------------------
    # Train
    # -------------------------
    model.train()
    model.backbone.eval()

    train_loss_total = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size_now = images.size(0)

        train_loss_total += loss.item() * batch_size_now

        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += batch_size_now

    train_loss = train_loss_total / train_total
    train_acc = train_correct / train_total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = loss_fn(logits, labels)

            batch_size_now = images.size(0)

            val_loss_total += loss.item() * batch_size_now

            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += batch_size_now

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    print(f"Epoch [{epoch + 1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print("-" * 50)

    # -------------------------
    # Save best classifier head
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "head_state_dict": model.head.state_dict(),
                "class_to_idx": train_dataset.class_to_idx,
                "image_size": image_size,
                "input_dim": 384,
                "num_classes": 2,
                "model_name": MODEL_NAME,
                "weights_name": WEIGHTS_NAME,
            },
            SAVE_PATH
        )

        print("Saved best classifier head.")

Class mapping:
{'symbiont_A': 0, 'symbiont_B': 1}


C:\Users\Wang2_h\dinov2\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\Wang2_h\dinov2\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'